# 03 - Final Inference

Applies the best trained model to the complete unseen dataset and
merges results with all human-labeled data into one annotated file.

In [ ]:
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
twits_folder        = BASE_PATH / 'Raw Data/Twits/'
test_folder         = BASE_PATH / 'Raw Data/'
datasets_folder     = BASE_PATH / 'Data Sets'
cleanedds_folder    = BASE_PATH / 'Data Sets/Cleaned Data'
networks_folder     = BASE_PATH / 'Data Sets/Networks/'
literature_folder   = BASE_PATH / 'Literature/'
topic_models_folder = BASE_PATH / 'Models/Topic Modeling/'
hitl_folder = datasets_folder / 'Classifiers_Data' / 'HITL'
out_folder  = datasets_folder / 'Classifiers_Data' / 'Final'
out_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
if not RUNNING_LOCALLY:
    print('Running Colab setup...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'transformers', 'torch'])
else:
    print('Running locally: skipping Colab setup.')

In [ ]:
import glob
import time
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

## 1. Load Best Model

In [ ]:
best_path = hitl_folder / 'best_roberta_model'
if not best_path.exists():
    raise FileNotFoundError('Model not found. Run notebook 02 first.')

tokenizer = AutoTokenizer.from_pretrained(str(best_path))
model     = AutoModelForSequenceClassification.from_pretrained(str(best_path))
device    = 0 if torch.cuda.is_available() else -1
clf_pipe  = pipeline('text-classification', model=model,
                     tokenizer=tokenizer, device=device, return_all_scores=True)
print('Model loaded.')

## 2. Load Inference Dataset

In [ ]:
df = pd.read_pickle(hitl_folder / 'inference_dataset.pkl')
print(f'Inference dataset: {len(df):,} tweets')

## 3. Run Inference

In [ ]:
preds = []
t0    = time.time()
for i in range(0, len(df), 500):
    preds.extend(clf_pipe(df['text'].iloc[i:i+500].astype(str).tolist()))
    if i % 50_000 == 0 and i > 0:
        print(f'  {i:,}/{len(df):,} processed...')
print(f'Inference done in {time.time()-t0:.1f}s')

df['predicted_label'] = [max(s, key=lambda x: x['score'])['label'] for s in preds]
df['confidence']      = [max(s, key=lambda x: x['score'])['score'] for s in preds]

## 4. Merge with Human Labels and Save

In [ ]:
labeled_files = sorted(glob.glob(str(hitl_folder / 'hitl_review_batch_*.csv')))
human_dfs = []
for f in labeled_files:
    tmp = pd.read_csv(f)
    if 'human_label' in tmp.columns:
        tmp['predicted_label'] = tmp['human_label'].fillna(tmp.get('predicted_label'))
        human_dfs.append(tmp.dropna(subset=['human_label']))

if human_dfs:
    human_df = pd.concat(human_dfs, ignore_index=True)
    human_df['is_human_labeled'] = True
else:
    human_df = pd.DataFrame()

df['human_label']      = np.nan
df['is_human_labeled'] = False

final_df = pd.concat([df, human_df], ignore_index=True)
print(f'Final dataset: {len(final_df):,} rows')

final_df.to_pickle(out_folder / 'final_annotated_tweets.pkl')
final_df.to_csv(out_folder / 'final_annotated_tweets.csv', index=False)
print('Saved to Classifiers_Data/Final/')